# 🎯 [PRC01_M01] 국방 조달 계약서, 입찰 공고문 및 방위사업법 규정 데이터 구조 분석
### 실습 솔루션 & 정답 해설 (Google Colab / Jupyter)

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/edunlab/milicontent2/blob/main/uploads/PRC01/prc01_m01_solution.ipynb)

---

## 📌 1. 프로젝트 및 미션 개요

| 항목 | 내용 |
| :--- | :--- |
| **과정 분류** | 조달·계약·국산화 (Level 3) |
| **미션 ID** | PRC01_M01 |
| **미션 주제** | 국방 조달 계약서, 입찰 공고문 및 방위사업법 규정 데이터 구조 분석 |
| **소요 시간** | 약 1.5시간 |
| **환경 사양** | Python 3.10+, numpy, pandas, matplotlib, scipy, scikit-learn |

### 📖 프로젝트 목표
조달계획안과 계약서는 지체상금률, 선금, 보증, 하도급, 필수 조항 같은 항목이 규정에 맞는지 사람이 한 건씩 읽고 대조해야 해서 검토가 늦어지고 불공정 조항이나 납기 지체 위험 문구를 놓치기 쉽다. 본 프로젝트는 조달계획안과 계약서의 오류와 위험을 검증하는 도구를 만드는 과제로, 조달 업무의 병목과 위험 신호 기준, 조항 수정 대안, 계약 의뢰 근거와 변경 이력, 업무 효과 측정까지 한 흐름으로 구현한다. 교육용 가상 계약 자료로 계약 데이터 스키마 정의, 서식별 자료 정제와 조항 정규화, 룰 기반 규정 준수 판별, 자연어처리 기반 독소·납기 위험 문구 식별, 위험 점수와 신뢰도 엔진, 조항 하이라이트 대시보드와 오류 처리, 계약관·법무 검토 HITL 워크플로우, 1,000건 일괄 스크리닝 부하 시험, AAR 룰셋 재보정, 통합 패키지까지 단계별로 만든다. 최종 목표는 위험 계약을 근거와 함께 먼저 보여 주어 검토 시간을 줄이되, 규정 위반 여부의 판단과 계약 승인은 계약관·법무·승인권자가 하도록 지원하는 도구를 완성하는 것이다.

> **⚠️ [교육용 가상 데이터셋 원칙]**
> - 본 노트북의 모든 데이터(제원, 로그, 시나리오 등)는 보안 규정을 준수하여 **코드 내 난수 기반 가상 데이터로 생성**됩니다.
> - 실제 부대 위치, 작전 인가, 실제 전장 정보, 군사 기밀 데이터는 일체 포함되지 않습니다.

---

### 📁 프로젝트 표준 산출물 디렉터리 체계 (`docs/`, `src/`, `dist/`) 안내
> **💡 [실무 프로토타입 업무 도구 구축 가이드]**
> 본 프로젝트는 단순 일회성 데이터 분석 실습이 아닌, 실제 군 부서(군수사령부, 조달본부, 계약관리과 등)에 배포 가능한 **업무 자동화 소프트웨어 도구(Tool)**를 단계별로 조립·완성하는 과정입니다. 실습 중 산출물(Deliverables)로 요구되는 파일들은 아래 표준 디렉터리에 모듈별로 저장·누적됩니다.
> - **`docs/` (국방 행정 및 기술 명세서):** 업무 정의서(`prc01_as_is_to_be_spec.md`), 스키마 명세서(`schema_definition.md`), UAT 검증 보고서(`prc01_uat_feedback_report.md`), 7개 장 운영 매뉴얼(`docs/prc01_final_manual.md`)
> - **`src/` (업무 도구 소스코드):** 데이터 파서(`connectors/`), 규정 준수 룰 엔진(`engine/`), NLP 지능 모듈(`ai/`), 대시보드 UI(`ui/`), 결함 검증기(`utils/`), HITL 워크플로우(`workflow/`), 배치 러너(`pipeline/`)
> - **`dist/` (배포 패키지):** 원클릭 자동 설치 스크립트(`dist/prc01_package/setup.sh`), 설치 가이드(`README.md`)
> *(주피터 노트북 환경에서는 각 미션 셀 검증 후 `%%writefile src/...` 매직 명령어를 활용하여 해당 모듈 파일로 자동 추출할 수 있습니다.)*

---


In [ ]:
# ============================================================================
# [Google Colab 환경 설정] 필수 라이브러리 설치 및 임포트
# ============================================================================
!pip install -q numpy pandas matplotlib scipy scikit-learn

import os
import sys
import time
import math
import json
import re
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings('ignore')

# 그래프 영문 폰트 기본 설정 (환경 독립성 보장)
plt.rcParams['font.sans-serif'] = 'DejaVu Sans'
plt.rcParams['axes.unicode_minus'] = False

print(f"실행 환경 준비 완료 (Python {sys.version.split()[0]})")


---

## 🎯 미션 01. 국방 조달 계약서, 입찰 공고문 및 방위사업법 규정 데이터 구조 분석
**소요시간**: 약 1.5시간

### 📋 미션 배경 및 수행 지침
#### 미션 배경 및 당위성

국방 조달 계약서, 입찰 공고문 및 방위사업법 규정 데이터 구조를 분석한다. 실제 전술 현장에서는 단순 탐지에 그치지 않고 신속한 위협 판단과 신뢰도 높은 의사결정이 필수적이므로, **조달계획안과 계약서(계약 방식·금액·기간·대금·보증 조건, 필수 조항 유무)의 데이터 구조와 입출력 명세를 정의하고, 현행(AS-IS) 조달계획안 검증 업무를 단계로 나누어 병목을 찾는다.** 과업을 완수하여 오판을 방지하고 작전 대응력을 극대화해야 합니다.

이 문제를 해결하기 위해 본 미션에서는 **계약 식별, 계약 방식·금액·기간, 지체상금률·선금·이행보증·하자보증, 하도급 비율, 필수 조항 유무를 묶어 필드 16개의 스키마와 검증 규칙 작성하여 원인을 분석하는** 실습을 체계적으로 수행합니다.

학습자는 실습가이드에 제시된 총 4개 세부 활동(기본 기능 구현 ➔ 결과 지표 검증 ➔ 파라미터 튜닝 및 도전과제)을 순차적으로 진행하며 핵심 판정 기준을 달성합니다.

**이번 결과물:** `docs/prc01_as_is_to_be_spec.md` 형태의 결과물을 완성한다.

### 🔍 평가 및 검증 기준 (PASS 조건)
#### 미션 판정

**평가 구분:** 필수

**핵심 평가영역:** 현행 업무(AS-IS)와 개선 업무(TO-BE) 비교의 명확성

#### 미션 PASS 조건

- 정량 기준: 업무 단계 분해 10단계 이상, 병목 구간 3개소 이상 도출


- 진행 기준: 업무 정의서 및 입출력 데이터 명세 작성 완료


- 검증 방법: 실무 프로세스와의 일치성 검증


#### 채점자 판정 메모

> 단순 코드 실행 여부뿐만 아니라 출력된 수치와 근거 자료가 군 실무 요구사항에 부합하는지 면밀히 확인한다.



In [ ]:
"""가상 조달계획안·계약서 자료(계약 속성 16개 + 조항 문장)의 스키마와 검증 규칙을 정의하고 오류 주입 시험으로 검출력을 확인한 뒤, 현행(AS-IS) 조달계획안 검증 업무를 12단계로 분해해 병목 구간과 개선(TO-BE) 효과를 비교한다."""
import re
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# ---------------------------------------------------------------------------
# [교육용가상자료] 가상 조달계획안·계약서 생성기 (실제 계약·업체·법령 조문 없음)
# 계약 속성 16개 + 조항 문장 6개를 만들고, 속성 위반·독소 조항·납기 위험 문구가 섞인 '규정 위반 여부'를 함께 만든다.
# ---------------------------------------------------------------------------
TYPES = ["물품", "용역", "공사"]
METHODS = ["일반경쟁", "제한경쟁", "지명경쟁", "수의계약"]
CLASSES = ["정상", "독소", "납기위험"]
OBJ = ["납품 물품", "용역 결과물", "시공 목적물", "정비 성과물"]

N_TPL = [
    "계약상대자가 납기를 지체한 경우 지체일수 1일마다 계약금액의 {p}/1000에 해당하는 지체상금을 부과한다.",
    "지체상금은 불가항력 사유가 확인된 경우에 한하여 발주기관의 승인을 받아 면제할 수 있다.",
    "납기 연장은 발주기관이 불가항력 사유를 확인하고 서면으로 승인한 경우에만 인정한다.",
    "발주기관은 납품된 {o}을 {d}일 이내에 검사하고 합격한 후에 대금을 지급한다.",
    "계약상대자는 하도급을 하려는 경우 사전에 발주기관의 서면 승인을 받아야 한다.",
    "어느 한쪽이 계약을 해지하려면 사유를 명시해 {d}일 전에 서면으로 통지해야 한다.",
    "{o}에 하자가 발생하면 계약상대자는 보증 기간 안에 무상으로 보수하거나 교체한다.",
    "계약 이행 중 알게 된 비공개 사항은 제3자에게 누설하지 않으며 위반 시 책임을 진다.",
    "{o}에 대한 지식재산권의 귀속과 사용 범위는 별도 합의서에 따라 정한다.",
    "대금은 검사 합격 후 청구서를 접수한 날부터 {d}일 이내에 지급한다.",
    "계약금액의 조정은 물가변동 기준에 따라 양 당사자가 서면으로 합의한 경우에만 한다.",
    "계약상대자는 이행보증서를 계약 체결 시 제출하며 보증금은 계약 이행 완료 후 반환한다.",
    "납품 일정은 계약서 별표의 일자를 따르며 변경 시 서면 합의가 필요하다.",
    "지체상금은 면제 사유가 없는 한 부과하며 이의는 정해진 절차로 제기할 수 있다.",
    "검사에 합격하지 못한 {o}은 계약상대자가 교체하며 그 비용은 계약상대자가 부담한다.",
    "계약상대자의 귀책 사유로 이행이 불가능하면 발주기관은 서면으로 통지하고 계약을 해지할 수 있다.",
    "지연이 예상되면 계약상대자는 사유와 만회 계획을 발주기관에 서면으로 보고한다.",
    "보증 기간 중 발생한 결함은 계약상대자가 책임지고 보수한다.",
    "분쟁은 먼저 협의로 해결하고 협의가 되지 않으면 관할 법원의 판단에 따른다.",
]
T_TPL = [
    "계약상대자의 지체상금은 전액 면제하며 발주기관은 이에 대해 이의를 제기하지 못한다.",
    "납품 즉시 검사에 합격한 것으로 보며 발주기관은 이후 검사를 요구하지 않는다.",
    "계약상대자는 언제든지 일방적으로 계약을 해지할 수 있으며 그에 따른 손해는 발주기관이 부담한다.",
    "{o}의 추가 비용은 사유를 불문하고 발주기관이 전액 부담하며 별도 협의 없이 청구할 수 있다.",
    "{o}에 관한 지식재산권은 모두 계약상대자에게 귀속되고 발주기관은 사용할 수 없다.",
    "하도급은 신고 없이 제한 없이 허용하며 발주기관은 하도급 내용을 확인하지 않는다.",
    "대금은 검사 전에 전액 선지급하며 발주기관은 반환 청구권을 포기한다.",
    "계약 이행 중 알게 된 사항의 누설로 인한 책임을 계약상대자는 지지 않는다.",
    "하자 보수 책임은 인도 후 즉시 소멸하며 발주기관은 보수를 요구할 수 없다.",
    "발주기관은 계약상대자가 청구하는 금액을 거절할 수 없으며 거절하면 위약금을 지급한다.",
    "분쟁이 생기면 계약상대자가 정하는 중재 기관에서만 해결하고 발주기관은 따른다.",
    "발주기관의 검사는 생략하고 계약상대자의 시험 결과서로 대신한다.",
    "견적 금액은 사후 정산 없이 그대로 확정하고 증빙 자료는 받지 않는다.",
]
D_TPL = [
    "납기는 자재 수급 사정에 따라 계약상대자가 연장할 수 있다.",
    "납품 일정은 가능한 한 지키도록 노력하며 지연되어도 사유를 보고하지 않는다.",
    "공급 지연이나 선행 업체 사정도 불가항력으로 보아 납기를 자동 연장한다.",
    "착수일은 선행 업체의 납품이 끝난 뒤 별도로 정하며 완료 기한은 정하지 않는다.",
    "납기일 이후 1회에 한해 기한 없이 자동 연장되는 것으로 한다.",
    "납기는 계약상대자의 생산 일정이 확정되는 대로 통보한다.",
    "납품 시점은 계약상대자 사정이 허락하는 때로 하고 날짜는 따로 정하지 않는다.",
    "자재가 늦게 들어오면 완료 일정은 그만큼 자동으로 늦춰진다.",
    "기한 내에 인도하지 못해도 계약상대자에게 책임을 묻지 않는다.",
    "일정이 늦어져도 계약상대자는 새 계획서를 낼 필요가 없다.",
]
# 학습에 쓰지 않는 '처음 보는 표현'(모델이 문장 틀을 외웠는지 확인하는 용도)
HOLD = {
    "정상": ["검사에 불합격한 경우 계약상대자는 지정 기한 안에 재납품하며 재검사 비용은 상대자가 부담한다.",
           "계약상대자가 법령을 위반하여 계약을 이행하지 못하면 발주기관은 서면 통지 후 계약을 해지할 수 있다.",
           "납기가 지연될 우려가 있으면 상대자는 즉시 사유와 만회 계획을 서면으로 보고해야 한다.",
           "하자담보 책임 기간은 인수일부터 기산하며 기간 중 발생한 결함은 상대자가 책임진다."],
    "독소": ["발주기관은 계약상대자가 요구하는 대금을 이유 없이 거절하지 못하며 거절 시 위약금을 낸다.",
           "계약 내용에 관한 분쟁은 계약상대자가 지정하는 기관에서만 다루며 발주기관은 이에 따른다.",
           "발주기관의 검수 권한은 포기하고 계약상대자의 자체 시험 성적서로 갈음한다.",
           "계약상대자가 제출한 견적은 사후 정산 없이 확정 단가로 하며 증빙은 요구하지 않는다."],
    "납기위험": ["납품 시기는 상대자의 형편이 허락하는 시점으로 하고 별도 날짜는 정하지 않는다.",
             "원자재 확보가 늦어지면 완료 일자는 자동으로 뒤로 미뤄지는 것으로 본다.",
             "인도 기한은 참고용이며 지키지 못해도 책임을 묻지 않는다.",
             "일정 지연이 있어도 상대자는 변경 계획을 제출할 의무가 없다."],
}
SUFFIX = ["", "", "", " 세부 사항은 시행 세칙에 따른다.", " 이 조항은 계약 전체에 적용한다.", " 필요한 경우 양 당사자가 협의한다.", " 관련 서식은 별지를 따른다."]
SWAP = [("발주기관", "계약담당관"), ("계약상대자", "업체")]


def make_clause(rng, cls, hold=False, idx=None):
    """조항 한 문장 생성. 표현 일부는 동의어로 바꾸고 접미 문구를 무작위로 붙인다(idx를 주면 그 번호의 문장 틀을 사용)."""
    if idx is not None:
        tpl = {"정상": N_TPL, "독소": T_TPL, "납기위험": D_TPL}[cls]
        s = tpl[idx].format(p=rng.choice(["1.0", "1.5", "2.0"]), d=int(rng.choice([7, 10, 14, 30])), o=rng.choice(OBJ))
    elif hold:
        s = HOLD[cls][rng.integers(len(HOLD[cls]))]
    else:
        tpl = {"정상": N_TPL, "독소": T_TPL, "납기위험": D_TPL}[cls]
        s = tpl[rng.integers(len(tpl))].format(p=rng.choice(["1.0", "1.5", "2.0"]), d=int(rng.choice([7, 10, 14, 30])), o=rng.choice(OBJ))
    if rng.random() < 0.3:
        a, b = SWAP[rng.integers(2)]
        s = s.replace(a, b)
    return s + SUFFIX[rng.integers(len(SUFFIX))]


def make_contracts(n=200, seed=7):
    """가상 계약 n건(속성 16개 + 조항 6개 + 규정 위반 정답)을 만든다."""
    rng = np.random.default_rng(seed)
    sl = rng.beta(1.2, 3.2, n)                                  # 계약서 작성이 허술한 정도(잠재 변수)
    ctype = rng.choice(TYPES, n, p=[.5, .3, .2])
    amount = np.clip(np.exp(rng.normal(np.log(220), 1.1, n)), 5, 5000).round(0)
    method = np.where(rng.random(n) < .06 + .3 * sl, "수의계약", rng.choice(METHODS[:3], n, p=[.5, .3, .2]))
    period = np.array([rng.integers(60, 300) if t == "물품" else rng.integers(90, 400) if t == "용역" else rng.integers(180, 720) for t in ctype])
    delivery = (period * rng.uniform(.5, 1.0, n)).astype(int)
    bad_logic = rng.random(n) < .015 * (1 + 3 * sl)
    delivery = np.where(bad_logic, period + rng.integers(10, 60, n), delivery)
    pen = np.where(rng.random(n) < .22 * sl, rng.choice([.3, .5, .8], n), rng.choice([1.0, 1.5, 2.0, 2.5], n))
    adv = np.where(rng.random(n) < .22 * sl, rng.choice([50, 60, 70], n), rng.choice([0, 10, 20, 30, 40], n))
    bond = np.where(rng.random(n) < .25 * sl + .5 * (adv > 40), rng.choice([0, 5], n), rng.choice([10, 10, 15, 20], n))
    war = np.array([rng.choice([3, 6]) if (rng.random() < .25 * s and t == "물품") else rng.choice([12, 24, 36]) if t == "물품" else rng.choice([0, 6, 12]) if t == "용역" else rng.choice([24, 36, 60]) for t, s in zip(ctype, sl)])
    insp = np.where(rng.random(n) < .3 * sl, rng.choice([21, 30], n), rng.choice([5, 7, 10, 14], n))
    sub = np.where(rng.random(n) < .15 * sl, rng.choice([60, 70, 80], n), rng.choice([0, 10, 20, 30, 40, 50], n))
    padj = rng.random(n) < .75 - .4 * sl
    term = rng.random(n) > .015 * (1 + 6 * sl)
    sec = rng.random(n) > .015 * (1 + 6 * sl)
    ip = rng.random(n) > .03 * (1 + 4 * sl)
    clauses, cls_true = [], []
    for i in range(n):
        pt, pd_ = .01 + .08 * sl[i], .02 + .08 * sl[i]
        cl = [rng.choice(CLASSES, p=[1 - pt - pd_, pt, pd_]) for _ in range(6)]
        cls_true.append(cl); clauses.append([make_clause(rng, c) for c in cl])
    df = pd.DataFrame({"contract_id": [f"C-{i + 1:04d}" for i in range(n)], "ctype": ctype, "method": method, "amount_m": amount,
                       "period_days": period, "delivery_days": delivery, "penalty_permil": pen, "advance_pct": adv, "perf_bond_pct": bond,
                       "warranty_months": war, "inspection_days": insp, "subcontract_pct": sub, "price_adjust": padj,
                       "termination": term, "security": sec, "ip_clause": ip})
    hard = ((pen < 1.0) | ((adv > 40) & (bond < 10)) | ((amount >= 300) & (method == "수의계약")) | ((ctype == "물품") & (war < 12))
            | ~term | ~sec | (sub > 50) | (delivery > period))
    toxic = np.array([c.count("독소") for c in cls_true])
    delay = np.array([c.count("납기위험") for c in cls_true])
    split = (method == "수의계약") & (amount >= 250) & (amount < 300) & (rng.random(n) < .6)      # 계약 쪼개기 의심(임계값 바로 아래)
    df["clauses"], df["clause_cls"] = clauses, cls_true
    df["n_toxic"], df["n_delay"] = toxic, delay
    viol = hard | (toxic > 0) | split | (delay >= 2)
    waived = hard & (toxic == 0) & (rng.random(n) < .12)                  # 계약관이 예외 사유를 인정해 위반으로 보지 않은 건(자료의 현실적 잡음)
    hidden = ~viol & (rng.random(n) < .03 + .08 * sl)                       # 속성·문구에 드러나지 않는 위반(별첨 합의 등)
    df["violation"] = (viol & ~waived) | hidden
    df["vtype"] = np.where(hidden, "기타", np.where(hard, "속성위반", np.where(toxic > 0, "독소조항", np.where(split | (delay >= 2), "기타", "-"))))
    return df

# 1) 스키마 정의: 계약 식별, 계약 방식·금액·기간, 대금·보증 조건, 필수 조항 유무
SCHEMA = {
    "contract_id": {"type": str, "pattern": r"^C-\d{4}$", "desc": "계약(계획안) 번호"},
    "ctype": {"type": str, "enum": TYPES, "desc": "계약 유형"},
    "method": {"type": str, "enum": METHODS, "desc": "계약 방식"},
    "amount_m": {"type": (int, float), "min": 1, "max": 100000, "desc": "계약 금액(백만원)"},
    "period_days": {"type": int, "min": 1, "max": 1500, "desc": "계약 기간(일)"},
    "delivery_days": {"type": int, "min": 1, "max": 1500, "desc": "납품 기한(착수 후 일수)"},
    "penalty_permil": {"type": (int, float), "min": 0, "max": 10, "desc": "지체상금률(1/1000)"},
    "advance_pct": {"type": (int, float), "min": 0, "max": 100, "desc": "선금 비율(%)"},
    "perf_bond_pct": {"type": (int, float), "min": 0, "max": 100, "desc": "이행보증 비율(%)"},
    "warranty_months": {"type": int, "min": 0, "max": 120, "desc": "하자보증 기간(개월)"},
    "inspection_days": {"type": int, "min": 1, "max": 90, "desc": "검사 기간(일)"},
    "subcontract_pct": {"type": (int, float), "min": 0, "max": 100, "desc": "하도급 비율(%)"},
    "price_adjust": {"type": bool, "desc": "물가변동 조항 유무"},
    "termination": {"type": bool, "desc": "해지 조항 유무"},
    "security": {"type": bool, "desc": "보안 조항 유무"},
    "ip_clause": {"type": bool, "desc": "지식재산 조항 유무"},
}


def check(name, v, r):
    if v is None or (isinstance(v, float) and np.isnan(v)):
        return [f"{name}: 필수 값 누락"]
    if not isinstance(v, r["type"]) or (isinstance(v, bool) and r["type"] is not bool):
        return [f"{name}: 자료형 오류({type(v).__name__})"]
    errs = []
    if "enum" in r and v not in r["enum"]:
        errs.append(f"{name}: 허용되지 않은 값 '{v}'")
    if "min" in r and v < r["min"]:
        errs.append(f"{name}: 최소값 {r['min']} 미만")
    if "max" in r and v > r["max"]:
        errs.append(f"{name}: 최대값 {r['max']} 초과")
    if "pattern" in r and not re.match(r["pattern"], v):
        errs.append(f"{name}: 형식 불일치 '{v}'")
    return errs


def validate(rec):
    errs = [f"{k}: 필드 누락" for k in SCHEMA if k not in rec]
    for k, r in SCHEMA.items():
        if k in rec:
            errs += check(k, rec[k], r)
    if not errs and rec["delivery_days"] > rec["period_days"]:                 # 필드 간 논리 규칙
        errs.append("delivery_days: 납품 기한이 계약 기간보다 김")
    if not errs and rec["method"] == "수의계약" and rec["amount_m"] > 5000:
        errs.append("method: 수의계약 금액이 비정상적으로 큼(5000백만원 초과)")
    return errs


def to_record(row):
    return {k: (row[k].item() if hasattr(row[k], "item") else row[k]) for k in SCHEMA}


con = make_contracts(100, seed=7)
records = [to_record(r) for _, r in con.iterrows()]
# 납품 기한이 계약 기간보다 긴 건은 '계약서 내용의 결함'이지 자료 형식 오류가 아니므로 형식 검증에서는 따로 센다
fmt_ok = sum(not [e for e in validate(r) if "납품 기한" not in e] for r in records)
logic = sum(any("납품 기한" in e for e in validate(r)) for r in records)
print(f"[스키마 검증] 계약 {len(records)}건 중 형식 통과 {fmt_ok}건, 형식 오류 {len(records) - fmt_ok}건 / 필드 {len(SCHEMA)}개 / 조항 문장 {sum(len(c) for c in con['clauses'])}개")
print(f"논리 규칙(납품 기한 > 계약 기간) 위반 {logic}건은 '계약 결함'으로 분류해 룰 엔진(미션 3)에서 다룬다")
print(f"수의계약 {int((con['method'] == '수의계약').sum())}건, 평균 금액 {con['amount_m'].mean():.0f}백만원, 지체상금률 1.0 미만 {int((con['penalty_permil'] < 1.0).sum())}건, 보안 조항 없음 {int((~con['security']).sum())}건")

# 2) 오류 주입 시험: 형식·허용값·범위·자료형·논리 규칙 위반 7종
base = records[1]
bad = dict(base, contract_id="C12", method="협의계약", amount_m=-5, period_days="90")
bad2 = dict(base, delivery_days=base["period_days"] + 30)
bad3 = dict(base, security=None)
bad4 = {k: v for k, v in base.items() if k != "penalty_permil"}
print("\n[오류 주입 결과]")
for e in validate(bad) + validate(bad2) + validate(bad3) + validate(bad4):
    print(" -", e)

# 3) AS-IS 업무 단계 분해: [교육용가상자료] 조달계획안 검증 30건의 단계별 소요일(작업 + 대기)
rng = np.random.default_rng(3)
STEPS = [("계획안·계약서 초안 접수", 1, 2), ("서식·첨부 서류 확인", 2, 3), ("사업·예산 부합 검토", 3, 2),
         ("계약 방식 적정성 검토", 2, 1), ("조항별 위험 문구 수작업 검토", 6, 2), ("규정 준수 점검표 작성", 4, 1),
         ("위험 조항 협의 회의", 1, 8), ("계약관·법무 검토 의뢰", 1, 9), ("수정 요청·업체 협의", 3, 7),
         ("수정본 재검토", 3, 2), ("결재 상신", 1, 5), ("계약 의뢰 등록", 1, 2)]
TOBE = {1: ("서식 자동 점검(미션 2)", 0.2), 4: ("위험 문구 자동 식별(미션 4)", 0.15), 5: ("룰 엔진 점검표 자동 생성(미션 3)", 0.1),
        6: ("위험 점수 기반 회의 안건 선별(미션 5)", 0.3), 9: ("수정 대조 자동 비교(미션 6)", 0.4)}
rows = []
for i, (name, work, wait) in enumerate(STEPS):
    w = rng.gamma(4, work / 4, 30); q = rng.gamma(2, wait / 2, 30)
    f = TOBE.get(i, ("유지(사람 판단)", 1.0))
    rows.append({"단계": f"{i + 1:02d}. {name}", "작업(일)": w.mean(), "대기(일)": q.mean(), "AS-IS(일)": w.mean() + q.mean(),
                 "TO-BE 방식": f[0], "TO-BE(일)": w.mean() * f[1] + q.mean() * (f[1] if f[1] < 1 else 1.0)})
proc = pd.DataFrame(rows)
share = proc["AS-IS(일)"] / proc["AS-IS(일)"].sum()
proc["병목"] = np.where((share >= share.sort_values().iloc[-3]) | (proc["대기(일)"] / proc["AS-IS(일)"] > 0.75), "병목", "")
print("\n[AS-IS / TO-BE 업무 단계 비교]")
print(proc.round(1).to_string(index=False))
n_bn = int((proc["병목"] == "병목").sum())
asis, tobe = proc["AS-IS(일)"].sum(), proc["TO-BE(일)"].sum()
print(f"\n업무 단계 {len(proc)}개(목표 10개 이상: {'달성' if len(proc) >= 10 else '미달'}), 병목 {n_bn}개소(목표 3개소 이상: {'달성' if n_bn >= 3 else '미달'})")
print(f"계획안 1건 검증 소요: AS-IS {asis:.1f}일 -> TO-BE {tobe:.1f}일 ({1 - tobe / asis:.0%} 단축 예상). 법무·계약관 검토와 결재는 사람이 수행한다.")

# 4) 입출력 데이터 명세 요약
print("\n[입출력 명세] 입력: 계약 속성 16개 필드 + 조항 문장 목록 / 출력: 위반 등급, 적중 룰·근거 조항, 위험 문구 하이라이트, 검토 이력")
print("[권한별 요구] 계약 담당: 입력·수정 / 계약관·법무: 검토·의견 / 승인권자: 결재(미션 7에서 구현)")

plt.figure(figsize=(9, 5))
y = np.arange(len(proc))
plt.barh(y - 0.2, proc["AS-IS(일)"], 0.4, label="AS-IS", color="tab:gray")
plt.barh(y + 0.2, proc["TO-BE(일)"], 0.4, label="TO-BE", color="tab:blue")
plt.yticks(y, [f"step {i + 1}{' *' if b else ''}" for i, b in enumerate(proc["병목"])])
plt.gca().invert_yaxis(); plt.xlabel("days per case"); plt.legend()
plt.title("Procurement plan review: AS-IS vs TO-BE (* = bottleneck, synthetic)")
plt.tight_layout(); plt.show()


<details>
<summary><b>💡 [참고] 미션 01 예상 실행 결과 (Output)</b></summary>

```text
[스키마 검증] 계약 100건 중 형식 통과 100건, 형식 오류 0건 / 필드 16개 / 조항 문장 600개
논리 규칙(납품 기한 > 계약 기간) 위반 2건은 '계약 결함'으로 분류해 룰 엔진(미션 3)에서 다룬다
수의계약 14건, 평균 금액 273백만원, 지체상금률 1.0 미만 2건, 보안 조항 없음 4건

[오류 주입 결과]
 - contract_id: 형식 불일치 'C12'
 - method: 허용되지 않은 값 '협의계약'
 - amount_m: 최소값 1 미만
 - period_days: 자료형 오류(str)
 - delivery_days: 납품 기한이 계약 기간보다 김
 - security: 필수 값 누락
 - penalty_permil: 필드 누락

[AS-IS / TO-BE 업무 단계 비교]
                  단계  작업(일)  대기(일)  AS-IS(일)                TO-BE 방식  TO-BE(일) 병목
   01. 계획안·계약서 초안 접수    1.0    2.0       3.0               유지(사람 판단)       3.0   
     02. 서식·첨부 서류 확인    2.6    2.8       5.4          서식 자동 점검(미션 2)       1.1   
     03. 사업·예산 부합 검토    3.1    2.2       5.3               유지(사람 판단)       5.3   
    04. 계약 방식 적정성 검토    2.3    1.0       3.3               유지(사람 판단)       3.3   
05. 조항별 위험 문구 수작업 검토    7.2    2.2       9.4       위험 문구 자동 식별(미션 4)       1.4 병목
    06. 규정 준수 점검표 작성    4.0    1.0       5.1    룰 엔진 점검표 자동 생성(미션 3)       0.5   
     07. 위험 조항 협의 회의    0.9    7.5       8.4 위험 점수 기반 회의 안건 선별(미션 5)       2.5 병목
    08. 계약관·법무 검토 의뢰    0.9    8.9       9.8               유지(사람 판단)       9.8 병목
     09. 수정 요청·업체 협의    3.3    6.7      10.0               유지(사람 판단)      10.0 병목
         10. 수정본 재검토    3.1    2.4       5.5       수정 대조 자동 비교(미션 6)       2.2   
           11. 결재 상신    0.9    5.7       6.6               유지(사람 판단)       6.6 병목
        12. 계약 의뢰 등록    1.0    2.1       3.1               유지(사람 판단)       3.1   

업무 단계 12개(목표 10개 이상: 달성), 병목 5개소(목표 3개소 이상: 달성)
계획안 1건 검증 소요: AS-IS 75.1일 -> TO-BE 49.0일 (35% 단축 예상). 법무·계약관 검토와 결재는 사람이 수행한다.

[입출력 명세] 입력: 계약 속성 16개 필드 + 조항 문장 목록 / 출력: 위반 등급, 적중 룰·근거 조항, 위험 문구 하이라이트, 검토 이력
[권한별 요구] 계약 담당: 입력·수정 / 계약관·법무: 검토·의견 / 승인권자: 결재(미션 7에서 구현)
```
</details>
